# Colab GPU 세션 준비

VS Code Colab 확장으로 Colab 서버에 연결해서 씁니다. 서버는 연결할 때마다 빈 머신에서 시작하므로, 새 런타임마다 이 셀들을 위에서부터 실행합니다.

1. 커널 선택 → **Colab** → GPU 런타임(T4 등) 선택
2. VS Code 탐색기에서 `.env`를 우클릭 → **Upload to Colab** (서버의 `/content/.env`로 올라감). `.env`에는 `DATA_URL`(데이터 zip의 Google Drive 공유 링크)이 있어야 합니다.
   링크는 노트북에 적지 않습니다(저장소가 public).

In [1]:
BRANCH = "main"  # 작업 중인 브랜치를 쓰려면 브랜치 이름으로 바꾼다

## 1. GPU 확인

In [2]:
import torch

assert torch.cuda.is_available(), "GPU 런타임이 아닙니다. 런타임 유형을 GPU로 바꾸세요."
print(torch.cuda.get_device_name(0))

Tesla T4


## 2. 저장소 받기

노트북은 VS Code(로컬)에서 열려 있어도 코드는 Colab 머신에서 실행되므로, 로컬의 프로젝트 파일을 import할 수 없습니다.
그래서 Colab 머신에 저장소를 받습니다. 같은 런타임이 살아 있으면 pull만 하고, 새 런타임이면 다시 clone합니다.
로컬에서 고친 코드는 push해야 Colab에 반영됩니다.

In [3]:
import importlib
import os
import sys
from pathlib import Path

REPO_DIR = Path("/content/pill-bbox")
if REPO_DIR.exists():
    !git -C {REPO_DIR} fetch -q origin {BRANCH} && git -C {REPO_DIR} checkout -q {BRANCH} && git -C {REPO_DIR} pull -q
else:
    !git clone -q -b {BRANCH} https://github.com/def-uk/pill-bbox.git {REPO_DIR}

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
# 같은 커널에서 새로 받은 .py를 import할 수 있도록 import 캐시를 비운다
importlib.invalidate_caches()
!git branch --show-current && git log --oneline -1

main
f7efbc3 (HEAD -> main, origin/main, origin/HEAD) Fix/colab data (#36)


## 3. 패키지 설치 (uv.lock 기준)

Colab 커널은 `.venv`가 아니라 Colab의 Python으로 돌아가므로, `uv sync` 대신 `uv.lock`에 고정된 버전을 커널의 Python에 그대로 설치합니다.
팀원 로컬 환경과 같은 버전이 깔립니다. torch는 lock에 적힌 CUDA 12.6 빌드로 바뀝니다.

In [4]:
if not Path("~/.local/bin/uv").expanduser().exists():
    !curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = f"{Path('~/.local/bin').expanduser()}:{os.environ['PATH']}"

!uv export -q --frozen --no-dev --no-hashes --no-emit-project -o /tmp/requirements.txt
!uv pip install --python {sys.executable} -r /tmp/requirements.txt --extra-index-url https://download.pytorch.org/whl/cu126 --index-strategy unsafe-best-match

downloading uv 0.13.5 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
Using Python 3.13.16 environment at: /usr
Resolved 106 packages in 913ms                                       
Prepared 58 packages in 1m 05s                                           
Uninstalled 41 packages in 1.10s
Installed 58 packages in 501ms                              
 - absl-py==2.0.0
 + absl-py==2.5.1
 + alembic==1.20.0
 - beautifulsoup4==4.13.5
 + beautifulsoup4==4.15.0
 - certifi==2026.10.19
 + certifi==2026.10.18
 - charset-normalizer==3.4.9
 + charset-normalizer==3.5.2
 + colorlog==6.12.0
 - cuda-bindings==13.0.3
 + cuda-bindings==12.9.10
 - cuda-toolkit==13.0.2
 + cuda-toolkit==12.6.3
 + cwevals==0.0.1b1
 - fsspec==2025.12.0
 + fsspec==2026.9.0
 - gdown==5.2.2
 + gdown==6.4.2
 + mako==1.4.3
 - markdown==3.11.2
 + markdown==3.11.1
 - matplotlib==3.10.0
 + matplotlib==3.11.2
 - numpy==2.1.3
 + numpy==2.5.4
 - nvidia-cublas-cu12==12.9.2.10
 + nvidia-cublas-cu12=

## 4. 환경 변수

코드는 Colab 서버에서 돌기 때문에 로컬(Mac)의 `.env`를 바로 읽을 수 없습니다.
VS Code 탐색기에서 `.env`를 우클릭 → **Upload to Colab** 하면 서버의 `/content/.env`로 올라가고, 이 셀이 그 파일을 읽습니다.
런타임이 새로 뜰 때마다 한 번 올립니다.

`/content/.env`에 없는 값은 Colab 보안 비밀에서 찾고, `DATA_URL`이 그래도 없으면 입력창으로 받습니다.
`WANDB_API_KEY`가 없으면 wandb를 끄고 진행합니다.

In [5]:
from getpass import getpass

from dotenv import load_dotenv

load_dotenv("/content/.env")  # 이미 설정된 환경 변수가 우선한다


def load_secret(name: str) -> str | None:
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return None


if not os.getenv("DATA_URL"):
    os.environ["DATA_URL"] = load_secret("DATA_URL") or getpass("DATA_URL: ")

if not os.getenv("WANDB_API_KEY"):
    wandb_key = load_secret("WANDB_API_KEY")
    if wandb_key:
        os.environ["WANDB_API_KEY"] = wandb_key
    else:
        os.environ["WANDB_MODE"] = "disabled"
        print("WANDB_API_KEY가 없어 wandb를 끄고 진행합니다.")
print("DATA_URL 설정됨:", bool(os.getenv("DATA_URL")))

WANDB_API_KEY가 없어 wandb를 끄고 진행합니다.
DATA_URL 설정됨: True


## 5. 데이터 받기

`DATA_URL`의 zip을 `data/raw`에 풉니다. 이미 파일이 있으면 건너뜁니다. 다시 받으려면 `overwrite=True`.

In [6]:
from data_process.download_data import download_data

download_data()

WARNING ⚠️ torchvision==0.29 is incompatible with torch==2.11.
Run 'pip install torchvision==0.26' to fix torchvision or 'pip install -U torch torchvision' to update both.
For a full compatibility table see https://github.com/pytorch/vision#installation
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


Downloading...
From (original): https://drive.google.com/uc?id=1Jjg_lxRPawCYpEKg0B4QerjdlUNQcKGG
From (redirected): https://drive.google.com/uc?id=1Jjg_lxRPawCYpEKg0B4QerjdlUNQcKGG&confirm=t&uuid=e5bad2bf-afd7-4530-a38a-0e45a0d71be9
To: /content/pill-bbox/data/tmpecu4tcw6/data.zip
100%|██████████| 1.93G/1.93G [00:27<00:00, 70.3MB/s]


/content/pill-bbox/data/raw: ['test_images', 'train_annotations', 'train_images']


PosixPath('/content/pill-bbox/data/raw')

## 6. 확인

In [7]:
from src.config import get_device

print(get_device())  # cuda가 나와야 한다

cuda
